# Prepare regions for upload to MorphoSource

We looked into uploading datasets for sharing, see https://github.com/habi/sticklebacks-manuscript/issues/11
We figured out, that [MorphoSource](https://www.morphosource.org/) is probably the best thing to try.
This notebook is used to prepare the `rec_regions` exports written by the `BucketSeparator.ipynb` notebook for upload to there.

The cells below are used to set up the whole notebook.
They load needed libraries and set some default values.

In [ ]:
# Load the modules we need
import platform
import os
import glob
import pandas
import dask
from tqdm.auto import tqdm

In [ ]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
# We use the fast internal SSD for speed reasons
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
from dask.distributed import Client
client = Client()

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
# Different locations if running either on Linux or Windows
FastSSD = True
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        BasePath = os.path.join('N:\\')
if 'research_storage_ben' in BasePath:
    Root = os.path.join(BasePath)
else:
    Root = os.path.join(BasePath, 'IEE Stickleback')
# Force reading from Bens research storage folder
# Root = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
print('We are loading all the data from %s' % Root)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files present on disk
# Using os.walk is way faster than using recursive glob.glob
# Not sorting the found logfiles is also making it quicker
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [ ]:
# Generate us a shorter folder name for easier printing
Data['FolderShort'] = [f[len(Root) + 1:] for f in Data['Folder']]

In [ ]:
# Show a (small) sampler of the loaded data as a first check
Data.sample(n=5)

In [ ]:
# Check for samples which are not yet reconstructed
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row.Folder and '_14/proj2' not in row.Folder and '_15/proj2' not in row.Folder:
        # Sticklebucket_14/proj2/Sticklebucket_14~00.log and 
        # Sticklebucket_15/proj2/Sticklebucket_15~00.log are failed scans where we cannot do a reconstruction, so we exclude them
        # If there's nothing with 'rec*' on the same level, then tell us
        if not glob.glob(row.Folder.replace('proj', '*rec*')):
            print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])

In [ ]:
# Search for any .csv files in each folder.
# These are only generated when the "X/Y Alignment With a Reference Scan" was performed in NRecon.
# If those files do *not* exist we have missed to do it and should correct for this.
Data['XYAlignment'] = [glob.glob(os.path.join(f, '*T*.csv')) for f in Data['Folder']]

In [ ]:
# Display samples which are missing the .csv-files for the XY-alignment
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row['Folder']:
        if not row['XYAlignment']:
            if not any(x in row.LogFile for x in ['rectmp.log',  # because we only exclude temporary logfiles in a later step
                                                  '14/proj',  # *All* scans of Sticklebucket 14 are missing the alignment files and cannot be aligned. 
                                                  '15/proj',  # *All* scans of Sticklebucket 15 are missing the alignment files and cannot be aligned
                                                  ]):
                print('- %s has *not* been X/Y aligned' % row.LogFile[len(Root) + 1:])

In [ ]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if 'ucket' not in row.Folder:  # Only use the scans named Bucket* here, e.g. BucketOfFish_* and Sticklebucket_*
        Data.drop([c], inplace=True)
    elif 'rec' not in row.Folder:  # Only look at logs in the rec folders
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in this notebook (to $scan$_region folders)
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith('._'):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '15um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '18um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '19um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'Sticklebucket_14' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)        
    elif 'Sticklebucket_15' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)                
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

It's a bit silly to exclude the self-written log files (`*_regions/*/*.log`) above, but like so we know for sure to include everything we've exported...

In [ ]:
# Generate us some meaningful colums in the dataframe
Data['Sample'] = [os.path.basename(log).replace('_rec.log', '') for log in Data['LogFile']]
Data['Scan'] = [os.path.basename(os.path.dirname(log)) for log in Data['LogFile']]

In [ ]:
# Show the data from the last loaded scans
Data.tail(n=5)

In [ ]:
# Load the file names of all the reconstructions of all the scans
Data['Filenames Reconstructions'] = [sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']]
# How many reconstructions do we have?
Data['Number of reconstructions'] = [len(r) for r in Data['Filenames Reconstructions']]

In [ ]:
# Drop samples which have either not been reconstructed yet or of which we deleted the reconstructions with
# `find . -name "*rec*.png" -type f -mtime +333 -delete`
# Based on https://stackoverflow.com/a/13851602
# for c,row in Data.iterrows():
#     if not row['Number of reconstructions']:
#         print('%s contains no PNG files, we might be currently reconstructing it' % row.Folder)
Data = Data[Data['Number of reconstructions'] > 0]
# Reset the dataframe count/index for easier indexing afterwards
Data.reset_index(drop=True, inplace=True)
print('We have %s folders with reconstructions' % (len(Data)))

In [ ]:
# Get parameters we need to submit to MorphoSource from the log files
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['ControlSoftware'] = [logparse.controlsoftware(log) for log in Data['LogFile']]
Data['Source'] = [logparse.source(log) for log in Data['LogFile']]
Data['Detector'] = [logparse.camera(log) for log in Data['LogFile']]
Data['DetectorVoxelsize'] = [logparse.cameravoxelsize(log) for log in Data['LogFile']]
Data['Voxelsize'] = [logparse.pixelsize(log) for log in Data['LogFile']]
Data['Filter'] = [logparse.whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [logparse.exposuretime(log) for log in Data['LogFile']]
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['NumProj'] = [logparse.numproj(log) for log in Data['LogFile']]

Data['Voltage'] = [logparse.voltage(log) for log in Data['LogFile']]
Data['Power'] = [logparse.power(log) for log in Data['LogFile']]
Data['Current'] = [logparse.current(log) for log in Data['LogFile']]

Data['Source object distance'] = [logparse.distance_source_to_sample(log) for log in Data['LogFile']]
Data['Source detector distance'] = [logparse.distance_source_to_detector(log) for log in Data['LogFile']]

Data['Averaging'] = [logparse.averaging(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [logparse.projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [logparse.rotationstep(log) for log in Data['LogFile']]
Data['Grayvalue'] = [logparse.reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [logparse.ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [logparse.beamhardening(log) for log in Data['LogFile']]
Data['DefectPixelMasking'] = [logparse.defectpixelmasking(log) for log in Data['LogFile']]
Data['Scan date'] = [logparse.scandate(log) for log in Data['LogFile']]

In [ ]:
# Sort dataframe based on the scan date
Data.sort_values(by=['Scan date'],
                 ignore_index=True,
                 inplace=True)

Since we've done everything *correctly* in `BucketSeparator.ipynb` we can simply go through all the desired folders and pull all in from disk.
This is more efficient than re-doing the extraction from scratch :)

In [ ]:
# Construct folder name for regions folder
Data['FolderRegionsExports'] = None
for c, row in Data.iterrows():
    Data.at[c, 'FolderRegionsExports'] = os.path.join(os.path.dirname(os.path.dirname(row.LogFile)), row.Scan + '_regions')

In [ ]:
# Search for log files we've written and construct the regions names from these
Data['RegionsLogFiles'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsLogFiles'] = sorted(glob.glob(os.path.join(row.FolderRegionsExports, '*', '*.log')))    

In [ ]:
# Construct regions name (and double-check for errors on the way)
Data['RegionsName'] = None
Data['RegionsFolder'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsName'] = [os.path.splitext(os.path.basename(logfilename))[0] for logfilename in row.RegionsLogFiles]
    Data.at[c, 'RegionsFolder'] = [os.path.dirname(logfilename) for logfilename in row.RegionsLogFiles]
    for rn, rf in zip(Data.at[c, 'RegionsName'], Data.at[c, 'RegionsFolder']):
        if rn != os.path.basename(rf):  # Tested with a manual rename on disk :)
            print(f'Error: For {row.LogFile[len(Root):]}: Extracted Region name "{rn}" does not match extracted folder name "{os.path.basename(rf)}"')

In [ ]:
# We should have 215+ regions, as this includes everything.
# Later we exclude 'WK.X24.001', as according to Ben, "[w]e also excluded Wik lake from all samples (as it had n =1)".
print('We have %s regions in total' % len(Data.RegionsName.explode()))

MorphoSource would like to ingest a ".zip containing .tif, .jpeg, .bmp, or .dcm*", see https://docs.google.com/document/d/1QByWl5t0SFD4QkdxUdoUbeTNms3HEhQYdTndo6PR-Ts/edit?tab=t.0, so we're preparing these files.
In [a test](https://www.morphosource.org/concern/media/000885110?locale=en), we've seen that a .zip with PNGs works fine, too...

Each .zip file should contain the original `proj/*.log`, `rec/*.log` and all the files from `rec_regions/FishID/*` for reproducible research.

In [ ]:
# Define us a "custom" zipping function
import pathlib
import zipfile

def zip_folder(folder, logfile, scan_date):
    # Generate folder names
    folder = pathlib.Path(folder)
    logfile = pathlib.Path(logfile)

    # Generate path for the zip file
    zip_path = folder.parent / (folder.name + ".zip")

    # Don't regenerate an existing ZIP
    if zip_path.exists():
        return zip_path

    # Search for correct label-checking file
    search_string = folder.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        folder.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )

    # Search for file in which we've specified the positions of the fish for the second batch of scans
    regionmdfile = next(
        folder.parent.parent.glob('*.Mapping*Region.md'),
        None
    )

    # Create README file (with function below)
    readme_path = create_readme(folder, logfile, scan_date)

    # Actually do the zipping now
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        # Add region folder
        for file in folder.rglob("*"):
            zf.write(file, arcname=file.relative_to(folder.parent))

        # Add reconstruction log file
        zf.write(logfile, arcname=logfile.name)

        # Add label-checking file
        zf.write(labelcheckingfile, arcname=labelcheckingfile.name)

        if regionmdfile is not None:
            # Add region metadata file
            zf.write(regionmdfile, arcname=regionmdfile.name)

        # Add README
        zf.write(readme_path, arcname="README.md")

    # Remove temporary README
    readme_path.unlink()

    return zip_path

In [ ]:
def get_file_listing(region):
    from pathlib import Path
   
    region = Path(region)

    files = sorted([f.relative_to(region) for f in region.rglob("*") if f.is_file()])

    pngs = [f for f in files if f.suffix.lower() == ".png"]
    regionlog = [f for f in files if f.suffix.lower() == ".log"]

    return pngs, regionlog

In [ ]:
# We want to add a custom/dynamic README.md file to *every* archive, so let's generate one
from datetime import datetime

def create_readme(region, logfile, scan_date):
    region = pathlib.Path(region)
    logfile = pathlib.Path(logfile)
    pngs, regionlog = get_file_listing(region)

    if len(pngs) > 2:
        png_listing = (
            f"│   ├── {pngs[0]}\n"
            f"│   ├── {pngs[1]}\n"
            f"│   ├── ...\n"
            f"│   ├── {pngs[-1]}"
        )
    elif len(pngs) == 1:
        png_listing = "\n".join(f"│   ├── {p}" for p in pngs)
    else:
        png_listing = ""

    log_listing = "\n".join(f"│   └── {l}" for l in regionlog)

    # Search for correct label-checking file
    search_string = region.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        region.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )    

    readme = f"""# {region.name}

## Info

This archive was generated on {datetime.now().isoformat(timespec="seconds")} with [a bespoke notebook](https://github.com/habi/sticklebacks-manuscript/PrepareForMorphoSource.ipynb) for uploading extracted datasets from [our sticklebacks manuscript](https://habi.github.io/sticklebacks-manuscript/) to [MorphoSource](https://www.morphosource.org/).
It contains the {len(pngs)} cropped reconstructions for specimen **{region.name}**, which was scanned on {scan_date}, and some aditional files.

## Archive contents

```console
{region.name}.zip
├── {region.name}/
{png_listing}
{log_listing}
├── {logfile.name}
├── {labelcheckingfile.name}
└── README.md
```

## Source

- `{region.name}`: Original region folder on disk from `{region.relative_to(Root)}`
- `{logfile.name}`: Log file of the reconstructions of the original multi-specimen scan copied into the archive from `{logfile.relative_to(Root)}`
- `{labelcheckingfile.name}`: Label/vial checking file generated from the the original multi-specimen scan. Generated with [the separator notebook](https://github.com/habi/sticklebacks/blob/main/BucketSeparator.ipynb) and copied into the archive from `{labelcheckingfile.relative_to(Root)}`.
- `README.md`: This file.
"""

    readme_path = region.parent / "README.md"
    readme_path.write_text(readme, encoding="utf-8")

    return readme_path

In [ ]:
for c, row in tqdm(Data.iterrows(), desc='Zipping', total=len(Data)):
    for d, region in tqdm(enumerate(row.RegionsFolder),
                          desc=f'Zipping regions of {row.FolderShort}',
                          total=len(row.RegionsFolder),
                          leave=False):
        if 'WK.X24.001' not in region: 
            # Skip the one specimen that was excluded from the manuscript because it was a singleton
            # According to Ben, "[w]e also excluded Wik lake from all samples (as it had n =1)".
            zip_folder(region, row.LogFile, row['Scan date'])
        else:
            print(f'Skipping {region} because it was excluded from the manuscript as a singleton')

In [ ]:
# How many zip files do we now have on disk?
Data['RegionsZipFiles'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsZipFiles'] = sorted(glob.glob(os.path.join(row.FolderRegionsExports, '*.zip')))

In [ ]:
print(f"We now have {Data['RegionsZipFiles'].explode().count()} zip files on disk.")

---
Now we have everything needed for MorphoSource.
If we want to batch-upload things to them, we need a "submission manifest Excel file (XLSX)", as mentioned on https://duke.atlassian.net/wiki/spaces/MD/pages/35423526/Batch+Submitting+Multiple+Media
For this we need some more data, which we pull from different sources into a new dataframe.

In [ ]:
MorphoSource = pandas.DataFrame()

In [ ]:
# Populate dataframe with the .zip files
MorphoSource['RegionsZipFiles'] = Data['RegionsZipFiles'].explode().reset_index(drop=True)
MorphoSource['File'] = [os.path.basename(fn) for fn in MorphoSource['RegionsZipFiles']]

In [ ]:
# Construct a fish ID column
MorphoSource['FishID'] = [os.path.splitext(os.path.basename(rzf))[0] for rzf in MorphoSource['RegionsZipFiles']]

In [ ]:
# Merge in original dataframe, exploded by the RegionsName, then we have all data we have at the moment  in the MorphoSource dataframe.
MorphoSource = MorphoSource.merge(Data.explode('RegionsName'), left_on='FishID', right_on='RegionsName', how='left')

In [ ]:
# MorphoSource would like to have the sex data.
# For the 2023 fish, this is unknown, for the 2024 fish, this is in `2024_Fish_Data_Lynn.csv`
# In there, we have the 'Unique ID' column, which is the same as the FishID column in our MorphoSource dataframe.
SexData = pandas.read_csv(os.path.join(Root, '2024_Fish_Data_Lynn.csv'))
# Merge the sex data into the MorphoSource dataframe
MorphoSource = MorphoSource.merge(SexData[['Unique_ID', 'Sex']], left_on='FishID', right_on='Unique_ID', how='left')

In [ ]:
# MorphoSource can also ingest lat/lon
# Accordint to Ben, this is
# --
# Watson: 60.539000, -150.467000
# Finger: 61.605600, -149.279200
# Spirit: 60.596104, -150.997664
# South Rolly: 61.667545, -150.135689
# Walby: 61.620000, -149.213000
# Tern: 60.533128, -149.547014
# --
# First construct the lake name (which we can add to MorphoSource as locality)
MorphoSource['LakeShort'] = [fishid.split('.')[0] for fishid in MorphoSource['FishID']]
lake_names = {
    "FG": "Finger Lake",
    "SL": "Spirit Lake",
    "SR": "South Rolly Lake",
    "WT": "Watson Lake",
    "TL": "Tern Lake",
    "WB": "Walby Lake",
}
MorphoSource['Lake'] = [lake_names.get(lake, 'Unknown') for lake in MorphoSource['LakeShort']]
# Construct lat/lon columns based on the lake name
lat_lon = {
    "Watson Lake": (60.539000, -150.467000),
    "Finger Lake": (61.605600, -149.279200),
    "Spirit Lake": (60.596104, -150.997664),
    "South Rolly Lake": (61.667545, -150.135689),
    "Walby Lake": (61.620000, -149.213000),
    "Tern Lake": (60.533128, -149.547014)
}
MorphoSource['LatLon'] = [lat_lon.get(lake, (0, 0)) for lake in MorphoSource['Lake']]
MorphoSource['Latitude'] = [latlon[0] for latlon in MorphoSource['LatLon']]
MorphoSource['Longitude'] = [latlon[1] for latlon in MorphoSource['LatLon']]

In [ ]:
# Populate MS dataframe with values that are equal for all scans
MorphoSource['Project'] = 'https://www.morphosource.org/projects/000885106'
MorphoSource['Species'] = 'Gasterosteus aculeatus'
MorphoSource['Object organization'] = 'https://www.morphosource.org/organizations/000898902'
MorphoSource['Creator'] = 'https://www.morphosource.org/users/b4a341'  # Used in 'imaging'
MorphoSource['Shading correction'] = True
MorphoSource['Surrounding material'] = 'Basotect (melamine resin foam)'
MorphoSource['Target type'] = 'Transmission'
MorphoSource['Detector type'] = 'Direct (X-Ray photocontuctor)'
MorphoSource['Detector configuration'] = 'Arae (single or tiled detector)'
MorphoSource['Target material'] = 'Tungsten'  # https://www.hamamatsu.com/content/dam/hamamatsu-photonics/sites/documents/99_SALES_LIBRARY/etd/L10711_TXPR1039E.pdf
MorphoSource['Rotation number'] = None
MorphoSource['Phase contrast'] = False
MorphoSource['Optical magnification'] = False
MorphoSource['Acquisition type'] = 'Sequenced (Rotational)'

In [ ]:
# Massage other data into the MorphoSource dataframe
# Imaging
MorphoSource.rename(columns={'Scan date': 'Event date'}, inplace=True)
MorphoSource['Software'] = [' '.join([scnr.replace(' ', ''), 'Control Program', f'(version {swv})']) for scnr, swv in zip(MorphoSource['Scanner'], MorphoSource['ControlSoftware'])]
MorphoSource.rename(columns={'Exposuretime': 'Exposure time'}, inplace=True)
MorphoSource.rename(columns={'Averaging': 'Frame averaging'}, inplace=True)
MorphoSource.rename(columns={'NumProj': 'Projections'}, inplace=True)
MorphoSource.rename(columns={'Current': 'Amperage'}, inplace=True)
MorphoSource.rename(columns={'Source': 'X-ray tube type'}, inplace=True)
MorphoSource['Detector pixels X'] = [ps[0] for ps in MorphoSource['ProjectionSize']]
MorphoSource['Detector pixels Y'] = [ps[1] for ps in MorphoSource['ProjectionSize']]
MorphoSource['Detector pixels size X'] = MorphoSource['DetectorVoxelsize']
MorphoSource['Detector pixels size Y'] = MorphoSource['DetectorVoxelsize']

The MorphoSource team asked us to *manually* upload 5 `.zip` files.
Surface the data for copy-pasting.

In [ ]:
fish_id_to_find = "SL.X24.001"

In [ ]:
for i, row in MorphoSource[MorphoSource['FishID'] == fish_id_to_find][[
    'FishID', 'Sex', 'Lake', 'Latitude', 'Longitude',
     'Scanner', 'Filter', 'Exposure time', 'Shading correction', 'Frame averaging',
    'Projections', 'Voltage', 'Power', 'Amperage', 'Surrounding material',
    'X-ray tube type', 'Target type', 'Detector type',
    'Detector pixels X', 'Detector pixels size X', 'Detector pixels Y', 'Detector pixels size Y',
    'Detector configuration', 'Source object distance', 'Source detector distance', 'Target material', 'Rotation number', 'Phase contrast', 'Optical magnification',
    'Acquisition type'
]].iterrows():

    print(f"\n--- {row['FishID']} ---")
    for key, value in row.items():
        print(f"{key:25}: {value}")

In [ ]:
# Write out (selected colums of) dataframe
MorphoSource.to_csv('MorphoSource_Export.csv', index=False)